In [0]:
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# GET ENVIRONMENT FROM ADF
# ============================================================

try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Using default: {env}")

# ============================================================
# STORAGE ACCOUNT
# ============================================================

storage_account_name = "capstonestorage01"

# ============================================================
# CONTAINER NAMES BASED ON ENVIRONMENT
# ============================================================

if env == 'DEV':
    gold_container = 'gold-dev'
elif env == 'TEST':
    gold_container = 'gold-test'
else:  # PROD
    gold_container = 'gold'

# ============================================================
# BUILD PATH
# ============================================================

GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

print("✅ Config ready!")

In [0]:
print("=" * 60)
print("  📊 DASHBOARD 1 — PORTFOLIO EXPOSURE")
print("=" * 60)

df = spark.read.format("delta")\
    .load(f"{GOLD}/fact_transactions")

# By Product
print("\n  💰 Total Outstanding by Product:")
df.groupBy("product_type")\
  .agg(
    F.round(F.sum("outstanding_balance"),2).alias("total_outstanding"),
    F.round(F.sum("exposure"),2).alias("total_exposure"),
    F.round(F.avg("utilization_pct"),2).alias("avg_utilization_pct"),
    F.countDistinct("account_id").alias("unique_accounts")
  ).orderBy("product_type").show()

# By Risk Segment
print("  ⚠️  Exposure by Risk Segment:")
df.groupBy("risk_segment")\
  .agg(
    F.round(F.sum("outstanding_balance"),2).alias("total_outstanding"),
    F.round(F.sum("exposure"),2).alias("total_exposure"),
    F.countDistinct("customer_id").alias("unique_customers")
  ).orderBy("risk_segment").show()

# By Year
print("  📅 Exposure Trend by Year:")
df.groupBy("year")\
  .agg(
    F.round(F.sum("outstanding_balance"),2).alias("total_outstanding"),
    F.round(F.sum("exposure"),2).alias("total_exposure")
  ).orderBy("year").show()

In [0]:
print("=" * 60)
print("  📊 DASHBOARD 2 — REVENUE ANALYSIS")
print("=" * 60)

# Interest & Fee by Product
print("\n  💵 Revenue by Product:")
df.groupBy("product_type")\
  .agg(
    F.round(F.sum("interest_accrued"),2).alias("total_interest"),
    F.round(F.sum("fee_amount"),2).alias("total_fees"),
    F.round(
        F.sum("interest_accrued") + F.sum("fee_amount"),2
    ).alias("total_revenue"),
    F.round(F.sum("repayment_amount"),2).alias("total_repayments"),
    F.count("transaction_id").alias("total_transactions")
  ).orderBy("product_type").show()

# Monthly Revenue Trend
print("  📅 Monthly Revenue Trend:")
df.groupBy("year","month")\
  .agg(
    F.round(
        F.sum("interest_accrued") + F.sum("fee_amount"),2
    ).alias("total_revenue"),
    F.count("transaction_id").alias("transactions")
  ).orderBy("year","month").show(48)

# Transaction Status
print("  📋 Transaction Status Breakdown:")
df.groupBy("transaction_status")\
  .agg(
    F.count("transaction_id").alias("count"),
    F.round(F.sum("amount"),2).alias("total_amount")
  ).orderBy("transaction_status").show()

In [0]:
print("=" * 60)
print("  📊 DASHBOARD 3 — DELINQUENCY & OVERDUE TRENDS")
print("=" * 60)

df_deliq = spark.read.format("delta")\
    .load(f"{GOLD}/fact_delinquency")

# DPD Bucket Summary
print("\n  ⚠️  DPD Bucket Summary:")
df_deliq.groupBy("dpd_bucket")\
  .agg(
    F.countDistinct("account_id").alias("accounts"),
    F.round(F.sum("overdue_amount"),2).alias("total_overdue"),
    F.round(F.avg("days_past_due"),2).alias("avg_dpd")
  ).orderBy("dpd_bucket").show()

# By Product Type
print("  📋 Delinquency by Product:")
df_deliq.groupBy("product_type","dpd_bucket")\
  .agg(
    F.countDistinct("account_id").alias("accounts"),
    F.round(F.sum("overdue_amount"),2).alias("total_overdue")
  ).orderBy("product_type","dpd_bucket").show()

# High Risk Customers by City
print("  🏙️  High Risk Overdue by City:")
df_deliq.filter(F.col("risk_segment") == "High")\
  .groupBy("city")\
  .agg(
    F.countDistinct("account_id").alias("high_risk_accounts"),
    F.round(F.sum("overdue_amount"),2).alias("total_overdue")
  ).orderBy(F.col("total_overdue").desc()).show()

In [0]:
print("=" * 60)
print("  📊 DASHBOARD 4 — CUSTOMER RISK SEGMENTATION")
print("=" * 60)

# Risk Segment Distribution
print("\n  👥 Risk Segment Distribution:")
df.groupBy("risk_segment")\
  .agg(
    F.countDistinct("customer_id").alias("total_customers"),
    F.round(F.sum("outstanding_balance"),2).alias("total_exposure"),
    F.round(F.avg("utilization_pct"),2).alias("avg_utilization")
  ).orderBy("risk_segment").show()

# Income Band vs Risk
print("  💰 Income Band vs Risk Segment:")
df.groupBy("risk_segment","income_band")\
  .agg(
    F.countDistinct("customer_id").alias("customers"),
    F.round(F.sum("outstanding_balance"),2).alias("exposure")
  ).orderBy("risk_segment","income_band").show()

# Age Group Analysis
print("  👤 Age Group Analysis:")
df.groupBy("age_group","risk_segment")\
  .agg(
    F.countDistinct("customer_id").alias("customers"),
    F.round(F.avg("utilization_pct"),2).alias("avg_utilization")
  ).orderBy("age_group").show()

In [0]:
print("=" * 60)
print("  📊 DASHBOARD 5 — BRANCH & CITY PERFORMANCE")
print("=" * 60)

# City Performance
print("\n  🏙️  Top Cities by Outstanding Balance:")
df.groupBy("city")\
  .agg(
    F.countDistinct("customer_id").alias("customers"),
    F.round(F.sum("outstanding_balance"),2).alias("total_outstanding"),
    F.round(F.sum("successful_amount"),2).alias("successful_amount"),
    F.count("transaction_id").alias("total_transactions")
  ).orderBy(F.col("total_outstanding").desc()).show()

# City vs Product
print("  📋 City vs Product Performance:")
df.groupBy("city","product_type")\
  .agg(
    F.round(F.sum("outstanding_balance"),2).alias("outstanding"),
    F.count("transaction_id").alias("transactions")
  ).orderBy(F.col("outstanding").desc()).show(20)

# Risk Band by City
print("  ⚠️  Risk Band Distribution by City:")
df.groupBy("city","risk_band")\
  .agg(
    F.countDistinct("account_id").alias("accounts")
  ).orderBy("city","risk_band").show()

In [0]:
print("=" * 60)
print("  📊 EXECUTIVE SUMMARY — PORTFOLIO HEALTH")
print("=" * 60)

total_exposure    = df.agg(F.sum("outstanding_balance")).collect()[0][0]
total_revenue     = df.agg(
    F.sum("interest_accrued") + F.sum("fee_amount")
).collect()[0][0]
total_customers   = df.select("customer_id").distinct().count()
total_accounts    = df.select("account_id").distinct().count()
total_txns        = df.count()
avg_utilization   = df.agg(F.avg("utilization_pct")).collect()[0][0]

df_deliq2 = spark.read.format("delta")\
    .load(f"{GOLD}/fact_delinquency")
total_overdue     = df_deliq2.agg(
    F.sum("overdue_amount")
).collect()[0][0]
delinquent_accs   = df_deliq2.filter(
    F.col("delinquency_flag") == 1
).select("account_id").distinct().count()

print(f"""
  ┌─────────────────────────────────────────────┐
  │         PORTFOLIO HEALTH SCORECARD          │
  ├─────────────────────────────────────────────┤
  │  Total Customers        : {total_customers:>15,}  │
  │  Total Accounts         : {total_accounts:>15,}  │
  │  Total Transactions     : {total_txns:>15,}  │
  │  Total Outstanding      : ₹{total_exposure:>14,.2f}  │
  │  Total Revenue          : ₹{total_revenue:>14,.2f}  │
  │  Avg Utilization        : {avg_utilization:>14.2f}%  │
  │  Total Overdue          : ₹{total_overdue:>14,.2f}  │
  │  Delinquent Accounts    : {delinquent_accs:>15,}  │
  └─────────────────────────────────────────────┘
""")
print("  ✅ DASHBOARD COMPLETE!")
print("=" * 60)

In [0]:
# Load all KPI tables as pandas
df_exposure  = spark.read.format("delta")\
    .load(f"{GOLD}/kpi_portfolio_exposure").toPandas()
df_revenue   = spark.read.format("delta")\
    .load(f"{GOLD}/kpi_revenue_analysis").toPandas()
df_deliq     = spark.read.format("delta")\
    .load(f"{GOLD}/kpi_delinquency_trends").toPandas()
df_risk      = spark.read.format("delta")\
    .load(f"{GOLD}/kpi_customer_risk").toPandas()
df_city      = spark.read.format("delta")\
    .load(f"{GOLD}/kpi_city_performance").toPandas()
df_fact      = spark.read.format("delta")\
    .load(f"{GOLD}/fact_transactions").toPandas()

print("✅ All data loaded!")
print(f"  exposure  : {len(df_exposure)} rows")
print(f"  revenue   : {len(df_revenue)} rows")
print(f"  deliq     : {len(df_deliq)} rows")
print(f"  risk      : {len(df_risk)} rows")
print(f"  city      : {len(df_city)} rows")

In [0]:
%pip install plotly

In [0]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

print("✅ Plotly imported successfully!")


In [0]:
df_exposure = spark.read.format("delta").load(f"{GOLD}/kpi_portfolio_exposure").toPandas()
df_revenue  = spark.read.format("delta").load(f"{GOLD}/kpi_revenue_analysis").toPandas()
df_deliq    = spark.read.format("delta").load(f"{GOLD}/kpi_delinquency_trends").toPandas()
df_risk     = spark.read.format("delta").load(f"{GOLD}/kpi_customer_risk").toPandas()
df_city     = spark.read.format("delta").load(f"{GOLD}/kpi_city_performance").toPandas()
df_fact     = spark.read.format("delta").load(f"{GOLD}/fact_transactions").toPandas()

print("✅ All data loaded!")

In [0]:
total_outstanding = df_fact["outstanding_balance"].sum()
total_revenue     = (df_fact["interest_accrued"] + df_fact["fee_amount"]).sum()
total_customers   = df_fact["customer_id"].nunique()
total_accounts    = df_fact["account_id"].nunique()
total_txns        = len(df_fact)
avg_utilization   = df_fact["utilization_pct"].mean()

fig = make_subplots(
    rows=1, cols=5,
    specs=[[
        {"type":"indicator"},
        {"type":"indicator"},
        {"type":"indicator"},
        {"type":"indicator"},
        {"type":"indicator"}
    ]]
)

fig.add_trace(go.Indicator(
    mode  = "number",
    value = total_customers,
    title = {"text": "👥 Total<br>Customers", "font": {"size": 18}},
    number= {"font": {"size": 40, "color": "#636EFA"}},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode  = "number",
    value = total_accounts,
    title = {"text": "🏦 Total<br>Accounts", "font": {"size": 18}},
    number= {"font": {"size": 40, "color": "#00CC96"}},
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode  = "number",
    value = round(total_outstanding/1e9, 2),
    title = {"text": "💰 Outstanding<br>(₹ Billions)", "font": {"size": 18}},
    number= {"font": {"size": 40, "color": "#EF553B"}, "suffix": "B"},
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode  = "number",
    value = round(total_revenue/1e6, 2),
    title = {"text": "💵 Total Revenue<br>(₹ Millions)", "font": {"size": 18}},
    number= {"font": {"size": 40, "color": "#FFA15A"}, "suffix": "M"},
), row=1, col=4)

fig.add_trace(go.Indicator(
    mode  = "number",
    value = round(avg_utilization, 2),
    title = {"text": "📈 Avg Utilization<br>%", "font": {"size": 18}},
    number= {"font": {"size": 40, "color": "#AB63FA"}, "suffix": "%"},
), row=1, col=5)

fig.update_layout(
    title = {
        "text"  : "📊 Executive Portfolio Scorecard — Financial Risk & Performance",
        "x"     : 0.5,
        "font"  : {"size": 24}
    },
    height   = 350,
    template = "plotly_dark",
    paper_bgcolor = "#1e1e2e",
    margin   = dict(t=100, b=50, l=50, r=50),
)
fig.show()

In [0]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "💰 Outstanding Balance by Product Type",
        "⚠️  Exposure by Risk Segment",
        "📈 Avg Utilization % by Product",
        "📅 Exposure Trend by Year"
    ),
    specs=[
        [{"type": "xy"},  {"type": "domain"}],
        [{"type": "xy"},  {"type": "xy"}]
    ],
    vertical_spacing   = 0.15,
    horizontal_spacing = 0.1
)

# Chart 1 — Bar: Outstanding by Product
prod = df_exposure.groupby("product_type")["total_outstanding"]\
    .sum().reset_index().sort_values("total_outstanding", ascending=False)
fig.add_trace(
    go.Bar(
        x=prod["product_type"],
        y=prod["total_outstanding"],
        marker_color=["#636EFA","#EF553B","#00CC96"],
        text=round(prod["total_outstanding"]/1e9, 2),
        texttemplate="%{text}B",
        textposition="outside",
        name="Outstanding"
    ), row=1, col=1
)

# Chart 2 — Pie: Exposure by Risk Segment
risk = df_exposure.groupby("risk_segment")["total_exposure"]\
    .sum().reset_index()
fig.add_trace(
    go.Pie(
        labels=risk["risk_segment"],
        values=risk["total_exposure"],
        hole=0.4,
        marker_colors=["#00CC96","#FFA15A","#EF553B"],
        textinfo="label+percent",
        textfont_size=14,
        name="Risk Exposure"
    ), row=1, col=2
)

# Chart 3 — Bar: Avg Utilization by Product
util = df_exposure.groupby("product_type")["avg_utilization_pct"]\
    .mean().reset_index()
fig.add_trace(
    go.Bar(
        x=util["product_type"],
        y=util["avg_utilization_pct"],
        marker_color=["#AB63FA","#FFA15A","#19D3F3"],
        text=round(util["avg_utilization_pct"], 1),
        texttemplate="%{text}%",
        textposition="outside",
        name="Utilization %"
    ), row=2, col=1
)

# Chart 4 — Line: Exposure Trend by Year
year = df_exposure.groupby("year")["total_exposure"]\
    .sum().reset_index().sort_values("year")
fig.add_trace(
    go.Scatter(
        x=year["year"].astype(str),
        y=year["total_exposure"],
        mode="lines+markers+text",
        text=round(year["total_exposure"]/1e9, 2),
        texttemplate="%{text}B",
        textposition="top center",
        line=dict(color="#FF6692", width=3),
        marker=dict(size=10),
        name="Yearly Exposure"
    ), row=2, col=2
)

fig.update_layout(
    title={
        "text" : "📊 Portfolio Exposure Dashboard",
        "x"    : 0.5,
        "font" : {"size": 24}
    },
    height        = 750,
    template      = "plotly_dark",
    paper_bgcolor = "#1e1e2e",
    plot_bgcolor  = "#1e1e2e",
    showlegend    = False,
    margin        = dict(t=120, b=60, l=60, r=60),
)

fig.update_xaxes(tickfont=dict(size=13))
fig.update_yaxes(tickfont=dict(size=13))

fig.show()

In [0]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "💵 Total Revenue by Product",
        "📊 Interest vs Fee Income",
        "📅 Monthly Revenue Trend",
        "💰 Repayments vs Disbursements"
    ),
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    vertical_spacing   = 0.15,
    horizontal_spacing = 0.1
)

# Chart 1 — Bar: Revenue by Product
rev_prod = df_revenue.groupby("product_type")["total_revenue"]\
    .sum().reset_index().sort_values("total_revenue", ascending=False)
fig.add_trace(
    go.Bar(
        x=rev_prod["product_type"],
        y=rev_prod["total_revenue"],
        marker_color=["#636EFA","#EF553B","#00CC96"],
        text=round(rev_prod["total_revenue"]/1e6, 1),
        texttemplate="%{text}M",
        textposition="outside",
        name="Revenue"
    ), row=1, col=1
)

# Chart 2 — Grouped Bar: Interest vs Fee
rev_type = df_revenue.groupby("product_type").agg({
    "total_interest_income": "sum",
    "total_fee_income"     : "sum"
}).reset_index()
fig.add_trace(
    go.Bar(
        x=rev_type["product_type"],
        y=rev_type["total_interest_income"],
        name="Interest",
        marker_color="#636EFA",
        text=round(rev_type["total_interest_income"]/1e6, 1),
        texttemplate="%{text}M",
        textposition="outside",
    ), row=1, col=2
)
fig.add_trace(
    go.Bar(
        x=rev_type["product_type"],
        y=rev_type["total_fee_income"],
        name="Fees",
        marker_color="#EF553B",
        text=round(rev_type["total_fee_income"]/1e6, 1),
        texttemplate="%{text}M",
        textposition="outside",
    ), row=1, col=2
)

# Chart 3 — Line: Monthly Revenue Trend
df_revenue["period"] = df_revenue["year"].astype(str) + "-" + \
                       df_revenue["month"].astype(str).str.zfill(2)
monthly = df_revenue.groupby("period")["total_revenue"]\
    .sum().reset_index().sort_values("period")
fig.add_trace(
    go.Scatter(
        x=monthly["period"],
        y=monthly["total_revenue"],
        mode="lines+markers",
        line=dict(color="#00CC96", width=2),
        marker=dict(size=6),
        name="Monthly Revenue"
    ), row=2, col=1
)

# Chart 4 — Grouped Bar: Repayments vs Disbursements
rd = df_revenue.groupby("product_type").agg({
    "total_repayments"   : "sum",
    "total_disbursements": "sum"
}).reset_index()
fig.add_trace(
    go.Bar(
        x=rd["product_type"],
        y=rd["total_repayments"],
        name="Repayments",
        marker_color="#AB63FA",
        text=round(rd["total_repayments"]/1e6, 1),
        texttemplate="%{text}M",
        textposition="outside",
    ), row=2, col=2
)
fig.add_trace(
    go.Bar(
        x=rd["product_type"],
        y=rd["total_disbursements"],
        name="Disbursements",
        marker_color="#FFA15A",
        text=round(rd["total_disbursements"]/1e6, 1),
        texttemplate="%{text}M",
        textposition="outside",
    ), row=2, col=2
)

fig.update_layout(
    title={
        "text" : "📊 Revenue Analysis Dashboard",
        "x"    : 0.5,
        "font" : {"size": 24}
    },
    height        = 750,
    template      = "plotly_dark",
    paper_bgcolor = "#1e1e2e",
    plot_bgcolor  = "#1e1e2e",
    barmode       = "group",
    margin        = dict(t=120, b=60, l=60, r=60),
)

fig.update_xaxes(tickfont=dict(size=12))
fig.update_yaxes(tickfont=dict(size=12))

fig.show()

In [0]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "⚠️  Accounts by DPD Bucket",
        "💸 Overdue Amount by Product",
        "🔴 Delinquency by Risk Segment",
        "📊 Average DPD by Product"
    ),
    specs=[
        [{"type": "xy"},     {"type": "xy"}],
        [{"type": "domain"}, {"type": "xy"}]
    ],
    vertical_spacing   = 0.15,
    horizontal_spacing = 0.1
)

# Chart 1 — Bar: Accounts by DPD Bucket
dpd = df_deliq.groupby("dpd_bucket")["delinquent_accounts"]\
    .sum().reset_index().sort_values("delinquent_accounts", ascending=False)
fig.add_trace(
    go.Bar(
        x=dpd["dpd_bucket"],
        y=dpd["delinquent_accounts"],
        marker_color=["#00CC96","#FFA15A","#EF553B","#AB63FA","#636EFA"],
        text=dpd["delinquent_accounts"],
        textposition="outside",
        name="Accounts"
    ), row=1, col=1
)

# Chart 2 — Bar: Overdue by Product
prod_overdue = df_deliq.groupby("product_type")["total_overdue"]\
    .sum().reset_index().sort_values("total_overdue", ascending=False)
fig.add_trace(
    go.Bar(
        x=prod_overdue["product_type"],
        y=prod_overdue["total_overdue"],
        marker_color=["#636EFA","#EF553B","#00CC96"],
        text=round(prod_overdue["total_overdue"]/1e6, 1),
        texttemplate="%{text}M",
        textposition="outside",
        name="Overdue"
    ), row=1, col=2
)

# Chart 3 — Pie: Delinquency by Risk Segment
risk_deliq = df_deliq.groupby("risk_segment")["delinquent_accounts"]\
    .sum().reset_index()
fig.add_trace(
    go.Pie(
        labels=risk_deliq["risk_segment"],
        values=risk_deliq["delinquent_accounts"],
        hole=0.4,
        marker_colors=["#00CC96","#FFA15A","#EF553B"],
        textinfo="label+percent",
        textfont_size=14,
        name="Risk"
    ), row=2, col=1
)

# Chart 4 — Bar: Avg DPD by Product
avg_dpd = df_deliq.groupby("product_type")["avg_dpd"]\
    .mean().reset_index()
fig.add_trace(
    go.Bar(
        x=avg_dpd["product_type"],
        y=avg_dpd["avg_dpd"],
        marker_color=["#AB63FA","#FFA15A","#19D3F3"],
        text=round(avg_dpd["avg_dpd"], 1),
        texttemplate="%{text} days",
        textposition="outside",
        name="Avg DPD"
    ), row=2, col=2
)

fig.update_layout(
    title={
        "text" : "📊 Delinquency & Overdue Trends Dashboard",
        "x"    : 0.5,
        "font" : {"size": 24}
    },
    height        = 750,
    template      = "plotly_dark",
    paper_bgcolor = "#1e1e2e",
    plot_bgcolor  = "#1e1e2e",
    showlegend    = False,
    margin        = dict(t=120, b=60, l=60, r=60),
)

fig.update_xaxes(tickfont=dict(size=12))
fig.update_yaxes(tickfont=dict(size=12))

fig.show()

In [0]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "👥 Risk Segment Distribution",
        "💰 Exposure by Income Band",
        "👤 Age Group vs Avg Utilization",
        "📊 Customers by Risk & Income Band"
    ),
    specs=[
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy"},     {"type": "xy"}]
    ],
    vertical_spacing   = 0.15,
    horizontal_spacing = 0.1
)

# Chart 1 — Pie: Risk Segment Distribution
risk_dist = df_risk.groupby("risk_segment")["total_customers"]\
    .sum().reset_index()
fig.add_trace(
    go.Pie(
        labels=risk_dist["risk_segment"],
        values=risk_dist["total_customers"],
        hole=0.4,
        marker_colors=["#00CC96","#FFA15A","#EF553B"],
        textinfo="label+percent",
        textfont_size=14,
        name="Risk"
    ), row=1, col=1
)

# Chart 2 — Bar: Exposure by Income Band
income = df_risk.groupby("income_band")["total_exposure"]\
    .sum().reset_index().sort_values("total_exposure", ascending=False)
fig.add_trace(
    go.Bar(
        x=income["income_band"],
        y=income["total_exposure"],
        marker_color=["#636EFA","#EF553B","#00CC96"],
        text=round(income["total_exposure"]/1e9, 2),
        texttemplate="%{text}B",
        textposition="outside",
        name="Exposure"
    ), row=1, col=2
)

# Chart 3 — Bar: Age Group vs Utilization
age = df_risk.groupby("age_group")["avg_utilization"]\
    .mean().reset_index().sort_values("age_group")
fig.add_trace(
    go.Bar(
        x=age["age_group"],
        y=age["avg_utilization"],
        marker_color=["#AB63FA","#FFA15A","#19D3F3","#FF6692"],
        text=round(age["avg_utilization"], 1),
        texttemplate="%{text}%",
        textposition="outside",
        name="Utilization %"
    ), row=2, col=1
)

# Chart 4 — Grouped Bar: Customers by Risk & Income
for seg, color in zip(
    ["High","Medium","Low"],
    ["#EF553B","#FFA15A","#00CC96"]
):
    subset = df_risk[df_risk["risk_segment"] == seg]\
        .groupby("income_band")["total_customers"].sum().reset_index()
    fig.add_trace(
        go.Bar(
            x=subset["income_band"],
            y=subset["total_customers"],
            name=seg,
            marker_color=color,
            text=subset["total_customers"],
            textposition="outside",
        ), row=2, col=2
    )

fig.update_layout(
    title={
        "text" : "📊 Customer Risk Segmentation Dashboard",
        "x"    : 0.5,
        "font" : {"size": 24}
    },
    height        = 750,
    template      = "plotly_dark",
    paper_bgcolor = "#1e1e2e",
    plot_bgcolor  = "#1e1e2e",
    barmode       = "group",
    margin        = dict(t=120, b=60, l=60, r=60),
)

fig.update_xaxes(tickfont=dict(size=12))
fig.update_yaxes(tickfont=dict(size=12))

fig.show()

In [0]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "🏙️  Top Cities by Outstanding Balance",
        "📊 Transactions by City",
        "🏦 City vs Product Performance",
        "👥 Customers by City"
    ),
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    vertical_spacing   = 0.15,
    horizontal_spacing = 0.1
)

city_sum = df_city.groupby("city").agg({
    "total_outstanding"  : "sum",
    "total_transactions" : "sum",
    "customers"          : "sum"
}).reset_index().sort_values("total_outstanding", ascending=False)

# Chart 1 — Bar: Outstanding by City
fig.add_trace(
    go.Bar(
        x=city_sum["city"],
        y=city_sum["total_outstanding"],
        marker_color=["#636EFA","#EF553B","#00CC96",
                      "#FFA15A","#AB63FA","#19D3F3","#FF6692"],
        text=round(city_sum["total_outstanding"]/1e9, 2),
        texttemplate="%{text}B",
        textposition="outside",
        name="Outstanding"
    ), row=1, col=1
)

# Chart 2 — Bar: Transactions by City
city_txn = city_sum.sort_values("total_transactions", ascending=False)
fig.add_trace(
    go.Bar(
        x=city_txn["city"],
        y=city_txn["total_transactions"],
        marker_color=["#EF553B","#636EFA","#00CC96",
                      "#FFA15A","#AB63FA","#19D3F3","#FF6692"],
        text=city_txn["total_transactions"],
        textposition="outside",
        name="Transactions"
    ), row=1, col=2
)

# Chart 3 — Grouped Bar: City vs Product
city_prod = df_city.groupby(["city","product_type"])["total_outstanding"]\
    .sum().reset_index()
for prod, color in zip(
    ["Loan","Credit Card","EMI"],
    ["#636EFA","#EF553B","#00CC96"]
):
    subset = city_prod[city_prod["product_type"] == prod]
    fig.add_trace(
        go.Bar(
            x=subset["city"],
            y=subset["total_outstanding"],
            name=prod,
            marker_color=color,
            text=round(subset["total_outstanding"]/1e9, 1),
            texttemplate="%{text}B",
            textposition="outside",
        ), row=2, col=1
    )

# Chart 4 — Bar: Customers by City
city_cust = city_sum.sort_values("customers", ascending=False)
fig.add_trace(
    go.Bar(
        x=city_cust["city"],
        y=city_cust["customers"],
        marker_color=["#00CC96","#636EFA","#EF553B",
                      "#FFA15A","#AB63FA","#19D3F3","#FF6692"],
        text=city_cust["customers"],
        textposition="outside",
        name="Customers"
    ), row=2, col=2
)

fig.update_layout(
    title={
        "text" : "📊 Branch & City Performance Dashboard",
        "x"    : 0.5,
        "font" : {"size": 24}
    },
    height        = 750,
    template      = "plotly_dark",
    paper_bgcolor = "#1e1e2e",
    plot_bgcolor  = "#1e1e2e",
    barmode       = "group",
    showlegend    = True,
    margin        = dict(t=120, b=60, l=60, r=60),
)

fig.update_xaxes(tickfont=dict(size=12))
fig.update_yaxes(tickfont=dict(size=12))

fig.show()

In [0]:
dbutils.notebook.exit("SUCCESS")